In [20]:
import numpy as np
import pandas as pd
import uuid
import random

In [21]:
# df

[Top 10 Vehicles in Malaysia](https://data.gov.my/dashboard/car-popularity)

[Guide to calculating insurance premium](https://bengkelbergerak.my/en/blog/kira-insurans-kereta)

[Premium calculator by Carso](https://www.carso.my/tool/car-insurance-calculator)

[Flooding risk weightage](https://www.dosm.gov.my/uploads/content-downloads/file_20220929154540.pdf)

In [22]:
features = [
    "POLID",

    # Plan details (required to obtain basic premium)
    "COVERAGE_TYPE",
    "VEHICLE_TYPE", # <- Not really needed for calculating premium, added for sum assured
    "CAR_AGE", # <- Not really needed for calculating premium, added for loadings as time progress
    "SUM_ASSURED",
    "REGION",
    "ENGINE_CAPACITY",

    # Insured details
    "DRIVER_AGE_CAT",
    "DRIVER_AGE",
    "DRIVER_GENDER",
    "MARITAL_STATUS",

    # Premium refinements
    "FLOOD_RISK",
    "THEFT_RISK",
    

    # Output from plan details
    "BASIC_PREMIUM",
    "FINAL_PREMIUM_SST",

    # Calculation specific variables
    "NCD_LEVEL",
    "NCD_YEARS",
    "COHORT_YEAR"
]

# Define dtypes for each column
dtype_dict = {
    "POLID": "string",

    # Plan details (required to obtain basic premium)
    "COVERAGE_TYPE": "category",
    "VEHICLE_TYPE": "category",
    "CAR_AGE": "int64",
    "SUM_ASSURED": "float64",
    "REGION": "category",
    "ENGINE_CAPACITY": "category",

    # Insured details
    "DRIVER_AGE_CAT": "category",
    "DRIVER_AGE": "int64",
    "DRIVER_GENDER": "category",
    "MARITAL_STATUS": "category",

    # Refinements
    "FLOOD_RISK": "boolean",
    "THEFT_RISK": "boolean",

    # Output from plan details
    "BASIC_PREMIUM": "float64",
    "FINAL_PREMIUM_SST": "float64",

    # Calculation specific variables
    "NCD_LEVEL": "int64",
    "NCD_YEARS": "int64",
    "COHORT_YEAR": "int64"
}

num_dataset = 10000

# Create DataFrame with specified dtypes
df = pd.DataFrame({col: pd.Series(dtype=dtype_dict[col]) for col in features})

# Fill POLID separately (since it needs a specific generation method)
df['POLID'] = [uuid.uuid4().hex for _ in range(num_dataset)]

print(df.dtypes)
print(f"\nDataFrame shape: {df.shape}")

POLID                     str
COVERAGE_TYPE        category
VEHICLE_TYPE         category
CAR_AGE               float64
SUM_ASSURED           float64
REGION               category
ENGINE_CAPACITY      category
DRIVER_AGE_CAT       category
DRIVER_AGE            float64
DRIVER_GENDER        category
MARITAL_STATUS       category
FLOOD_RISK            boolean
THEFT_RISK            boolean
BASIC_PREMIUM         float64
FINAL_PREMIUM_SST     float64
NCD_LEVEL             float64
NCD_YEARS             float64
COHORT_YEAR           float64
dtype: object

DataFrame shape: (10000, 18)


In [ ]:
# For Calculating basic premium

# Coverage Type
comprehensive_pct = 0.80
tpo_pct = 0.20

df["COVERAGE_TYPE"] = random.choices(
    ["Comprehensive", "TPO"],
    weights = (comprehensive_pct, tpo_pct),
    k = num_dataset
)

# Vehicle Type
ice_pct = 0.90
ev_pct = 0.10

df["VEHICLE_TYPE"] = random.choices(
    ["ICE", "EV"],
    weights = (ice_pct, ev_pct),
    k = num_dataset
)

# CAR_AGE feature: vehicle age at policy inception (0-10 years)
# Overall median ~3-4 years; Gen-Z skew to newer cars, older cohorts drive older cars

CAR_AGE_MEDIAN = {
    "Gen-Z": 2.0,
    "Millennial": 3.5,
    "Boomers": 5.0,
    "Senior": 5.5,
}

def generate_car_age(age_cat):
    """Draw vehicle age in [0, 10] for a driver age category."""
    base = CAR_AGE_MEDIAN[age_cat]
    return int(np.clip(round(base + np.random.normal(0, 1.5)), 0, 10))

def generate_driver_age(age_cat):
    """Fresh driver age draw by category (used for aging + new entrants)."""
    if age_cat == "Gen-Z":
        return np.random.randint(18, 28)
    if age_cat == "Millennial":
        return np.random.randint(28, 46)
    if age_cat == "Boomers":
        return np.random.randint(46, 66)
    return np.random.randint(66, 76)

def age_to_cat(age):
    """Map an exact age back to its driver age category."""
    if age <= 27:
        return "Gen-Z"
    if age <= 45:
        return "Millennial"
    if age <= 65:
        return "Boomers"
    return "Senior"

# Sum assured
ice_mask = df["VEHICLE_TYPE"] == "ICE"
ev_mask = df["VEHICLE_TYPE"] == "EV"

df.loc[ice_mask, "SUM_ASSURED"] = np.random.lognormal(
    mean = np.log(50000),  # Log of median
    sigma = 0.5,            # Spread
    size = sum(ice_mask)
)

# Log-normal for EV (median ~ RM 90k)
df.loc[ev_mask, "SUM_ASSURED"] = np.random.lognormal(
    mean = np.log(80000),  # Log of median
    sigma = 0.5,            # Spread
    size = sum(ev_mask)
)

# Round to nearest RM 1000
df["SUM_ASSURED"] = np.round(df["SUM_ASSURED"] / 1000) * 1000

# Region split
peninsular_my_pct = 0.80
east_my_pct = 0.20

df["REGION"] = random.choices(
    ["Peninsular Malaysia", "East Malaysia (Sabah, Sawarak & Labuan)"],
    weights = (peninsular_my_pct, east_my_pct),
    k = num_dataset
)

# Engine Capacity split
exp_weights = lambda lam, n=8, seed=None: (  # noqa: PLC3002
    lambda s: s / s.sum()
)(np.sort(np.random.default_rng(seed).exponential(1/lam, n))[::-1])

df["ENGINE_CAPACITY"] = random.choices(
    [
        "0 to 1,400 cc / EV up to 70 kW",
        "1,401 to 1,650 cc / EV 71 - 100 kW",
        "1,651 - 2,200 cc / EV 101 - 125 kW",
        "2,201 - 3,050 cc / EV 126 - 150 kW",
        "3,051 - 4,100 cc / EV 151 - 200 kW",
        "4,101 - 4,250 cc / EV 201 - 250 kW",
        "4,251 - 4,400 cc / EV 251 - 300 kW",
        "Over 4,400 cc / EV > 300 kW"
    ],
    weights = exp_weights(lam=1.0, seed=42),
    k = num_dataset
)

In [24]:
# Person insured details

# Person age
gen_z_pct = 0.40
millennial_pct = 0.40
boomer_pct = 0.15
senior_pct = 0.05

df["DRIVER_AGE_CAT"] = random.choices(
    ["Gen-Z", "Millennial", "Boomers", "Senior"],
    weights = (gen_z_pct, millennial_pct, boomer_pct, senior_pct),
    k = num_dataset
)

def generate_age_by_category(category):
    if category == "Gen-Z":
        return np.random.randint(18, 28)  # 18-27 (since upper bound is exclusive)
    elif category == "Millennial":
        return np.random.randint(28, 46)  # 28-45
    elif category == "Boomers":
        return np.random.randint(46, 66)  # 41-65
    else:  # Silent Generation or others
        return np.random.randint(66, 76)  # 65-75

df["DRIVER_AGE"] = df["DRIVER_AGE_CAT"].apply(generate_age_by_category)

# Gender distribution
male_pct = 0.55
female_pct = 0.45

df["DRIVER_GENDER"] = random.choices(
    ["Male", "Female"],
    weights = (male_pct, female_pct),
    k = num_dataset
)

def generate_age_by_category(age_cat):
    """
    Generate marital status based on age category with realistic probabilities
    """
    if age_cat == "Gen-Z":
        return np.random.choice(
            ['Single', 'Married'],
            p=[0.85, 0.15]
        )
    
    elif age_cat == "Millennial":
        return np.random.choice(
            ['Single', 'Married'],
            p=[0.50, 0.50]
        )
    else:  # Silent Generation
        return np.random.choice(
            ['Single', 'Married'],
            p=[0.20, 0.80]
        )

df["MARITAL_STATUS"] = df["DRIVER_AGE_CAT"].apply(generate_age_by_category)

In [25]:
peninsular_mask = df["REGION"] == "Peninsular Malaysia"
east_mask = df["REGION"] == "East Malaysia (Sabah, Sawarak & Labuan)"

# Flood Risk by Region
# Peninsular Malaysia: 60% flood risk, 40% no risk
# East Malaysia: 0% flood risk (assumed no flood risk)


# Peninsular: 60% True, 40% False
df.loc[peninsular_mask, "FLOOD_RISK"] = np.random.choice(
    [True, False], 
    size=peninsular_mask.sum(), 
    p=[0.60, 0.40]
)

# East Malaysia: 100% False
df.loc[east_mask, "FLOOD_RISK"] = False

# Theft Risk by Region
# Peninsular Malaysia: higher urban crime rates
# East Malaysia: lower rates overall

# Peninsular: 40% True, 60% False
df.loc[peninsular_mask, "THEFT_RISK"] = np.random.choice(
    [True, False], 
    size=peninsular_mask.sum(), 
    p=[0.40, 0.60]
)

# East Malaysia: 15% True, 85% False
df.loc[east_mask, "THEFT_RISK"] = np.random.choice(
    [True, False], 
    size=east_mask.sum(), 
    p=[0.15, 0.85]
)

In [26]:
# NCD (No Claim Discount) setup - exact table progression
# Malaysia PIAM NCD table per year
NCD_TABLE = {0: 0.00, 1: 0.25, 2: 0.30, 3: 0.3833, 4: 0.45, 5: 0.55}
COHORT_YEAR = 2026

df['NCD_LEVEL'] = 0.0
df['NCD_YEARS'] = 0
df['COHORT_YEAR'] = COHORT_YEAR

print(f'NCD setup complete. {len(df)} policies initialized at 0% NCD')
df[['POLID', 'NCD_LEVEL', 'NCD_YEARS', 'COHORT_YEAR']].head()

NCD setup complete. 10000 policies initialized at 0% NCD


,POLID,NCD_LEVEL,NCD_YEARS,COHORT_YEAR
0,68ffa4079a8f444f88e89adb1a8dd94a,0.0,0,2026
1,ffe93c5c08d74aed884d87c3260fc3ee,0.0,0,2026
2,fc69878caf2c448ba59ebcc0870d0237,0.0,0,2026
3,062c1be1fc134f008a00adc52dbb6de9,0.0,0,2026
4,197fac5aead54cd5852b52ec2319262d,0.0,0,2026


In [27]:
# Load the rates CSV once (outside the function for efficiency)
df_rates = pd.read_csv('./sources/rates.csv', index_col="Index")

def calculate_premium(row):
    key = f"{row['COVERAGE_TYPE']}_{row['REGION']}_{row['ENGINE_CAPACITY']}"

    # Check coverage type
    if row['COVERAGE_TYPE'] == "Comprehensive":
        rate = df_rates.loc[key, 'Rate']
        return np.round((row['SUM_ASSURED'] * rate), 2)

    if row['COVERAGE_TYPE'] == "TPO":
        premium = df_rates.loc[key, 'Fixed Amount']
        return np.round(premium, 2)

# Apply to each row
df["BASIC_PREMIUM"] = df.apply(calculate_premium, axis=1)


In [28]:

# Rating loadings: driver age category + vehicle age
# Driver loading: Gen-Z highest (inexperience); experienced cohorts lower.
# Car loading: increases linearly with vehicle age (older car = higher risk).

DRIVER_AGE_LOADING = {
    "Gen-Z": 1.20,
    "Millennial": 1.05,
    "Boomers": 1.00,
    "Senior": 1.05,
}

def driver_age_loading(age_cat):
    return DRIVER_AGE_LOADING.get(age_cat, 1.00)

def car_age_loading(car_age):
    """Linear vehicle-age loading; car age capped at 10 years."""
    return 1 + 0.03 * min(int(car_age), 10)

def total_loading(age_cat, car_age):
    """Combined driver x vehicle loading applied to the premium."""
    return driver_age_loading(age_cat) * car_age_loading(car_age)

print('Rating loadings defined:')
print(f"  Driver: {DRIVER_AGE_LOADING}")
print("  Car: 1 + 0.03 x CAR_AGE (capped at 10 yrs)")
print(f"  Example Gen-Z w/ 3-yr car: {total_loading('Gen-Z', 3):.3f}")


Rating loadings defined:
  Driver: {'Gen-Z': 1.2, 'Millennial': 1.05, 'Boomers': 1.0, 'Senior': 1.05}
  Car: 1 + 0.03 x CAR_AGE (capped at 10 yrs)
  Example Gen-Z w/ 3-yr car: 1.308


In [29]:

# FINAL_PREMIUM_SST: BASIC x driver/car loading x (1 - NCD) x 1.1^risk flags + 8% SST
# NCD discount applies to ALL coverages (both Comprehensive and TPO).
SST_RATE = 0.08  # Changeable variable - current SST rate in Malaysia

def compute_final_premium(row, sst_rate=SST_RATE):
    """Compute final premium with rating loadings, NCD discount, risk multipliers, SST."""
    loading = total_loading(row['DRIVER_AGE_CAT'], row['CAR_AGE'])
    ncd_discount = 1 - row.get('NCD_LEVEL', 0.0)
    risk_multiplier = 1.1 ** (int(row['FLOOD_RISK']) + int(row['THEFT_RISK']))
    final = row['BASIC_PREMIUM'] * loading * ncd_discount * risk_multiplier * (1 + sst_rate)
    return round(float(final), 2)

df['FINAL_PREMIUM_SST'] = df.apply(compute_final_premium, axis=1)
df['TOTAL_LOADING'] = df.apply(
    lambda r: total_loading(r['DRIVER_AGE_CAT'], r['CAR_AGE']), axis=1
)

sample = df[['POLID', 'BASIC_PREMIUM', 'DRIVER_AGE_CAT', 'CAR_AGE', 'NCD_LEVEL',
             'FLOOD_RISK', 'THEFT_RISK', 'FINAL_PREMIUM_SST']].head(10)
print('Premium calculation complete:')
print(sample.to_string())
print(f"\nAvg basic premium: RM{df['BASIC_PREMIUM'].mean():.2f}")
print(f"Avg final premium: RM{df['FINAL_PREMIUM_SST'].mean():.2f}")
print(f"Avg total loading: {df['TOTAL_LOADING'].mean():.3f}")


ValueError: cannot convert float NaN to integer

In [ ]:

# Claim Frequency Model (Poisson GLM, log-linear)
# lambda = exp(log_lambda) * coverage_multiplier
# Base exp(-2.00) ~ 0.135 claims/year - realistic Malaysian market level

def compute_claim_lambda(row):
    """Compute Poisson rate lambda via log-linear rating model."""
    log_lambda = -2.00  # baseline: ~13.5% annual claim frequency

    cat = row['DRIVER_AGE_CAT']
    if cat == 'Gen-Z':
        log_lambda += 0.40        # young drivers: higher risk
    elif cat == 'Senior':
        log_lambda += 0.26        # seniors: moderate increase

    # Young male interaction
    if cat == 'Gen-Z' and row['DRIVER_GENDER'] == 'Male':
        log_lambda += 0.05

    # EV proxy (higher power/repair exposure)
    if row['VEHICLE_TYPE'] == 'EV':
        log_lambda += 0.05

    # Risk flags
    if row['FLOOD_RISK']:
        log_lambda += 0.20
    if row['THEFT_RISK']:
        log_lambda += 0.10

    # Vehicle age: older cars carry higher breakdown/repair frequency
    log_lambda += 0.03 * row['CAR_AGE']

    # NCD safety credit: claim-free drivers are safer
    log_lambda -= 0.05 * row['NCD_YEARS']

    freq = np.exp(log_lambda)

    # Coverage multiplier: TPO has no own-damage exposure
    mult = 0.45 if row['COVERAGE_TYPE'] == 'TPO' else 1.00
    return freq * mult


df['CLAIM_LAMBDA'] = df.apply(compute_claim_lambda, axis=1)

print('Claim frequency model (Poisson GLM) applied')
print(f"Mean lambda: {df['CLAIM_LAMBDA'].mean():.4f}")
print(f"Min lambda: {df['CLAIM_LAMBDA'].min():.4f}, "
      f"Max lambda: {df['CLAIM_LAMBDA'].max():.4f}")
print(f"TPO mean lambda: {df.loc[df['COVERAGE_TYPE']=='TPO', 'CLAIM_LAMBDA'].mean():.4f}")
print(f"Comp mean lambda: {df.loc[df['COVERAGE_TYPE']=='Comprehensive', 'CLAIM_LAMBDA'].mean():.4f}")


In [ ]:
# Claim Severity Model: Per-Peril Gamma with Policy Caps
# Peril mix follows Malaysian retail product structure:
#   Comprehensive: own damage (AD/Windscreen/Theft/Fire) + third party (TPPD/TPBI)
#   TPO: third party only (TPPD/TPBI) - structurally cheaper claims

PERIL_DIST = {
    'Comprehensive': {
        'AD': 0.58, 'Windscreen': 0.15, 'Theft': 0.08,
        'Fire': 0.04, 'TPPD': 0.12, 'TPBI': 0.03
    },
    'TPO': {
        'TPPD': 0.78, 'TPBI': 0.22
    }
}

PERIL_BASE = {
    'TPBI':       {'shape': 0.35, 'scale': 70000, 'cap': float('inf')},
    'TPPD':       {'shape': 0.55, 'scale': 9000,  'cap': 3000000},
    'Windscreen': {'shape': 2.00, 'scale': 700,   'cap': 15000}
}


def sample_claim_peril(coverage_type):
    """Sample a claim peril from the product-specific mix."""
    mix = PERIL_DIST.get(coverage_type, PERIL_DIST['Comprehensive'])
    return np.random.choice(list(mix.keys()), p=list(mix.values()))


def generate_single_claim(coverage_type, sum_assured):
    """Draw one claim amount (RM) with peril-specific Gamma + cap."""
    peril = sample_claim_peril(coverage_type)

    if peril == 'Theft':
        shape, scale = 1.10, max(8000, min(20000, sum_assured * 0.20))
        cap = sum_assured
    elif peril == 'Fire':
        shape, scale = 0.90, max(7000, min(18000, sum_assured * 0.15))
        cap = sum_assured
    elif peril == 'AD':
        shape, scale = 0.60, max(4500, min(12000, sum_assured * 0.10))
        cap = sum_assured
    else:
        spec = PERIL_BASE[peril]
        shape, scale, cap = spec['shape'], spec['scale'], spec['cap']

    amount = np.random.gamma(shape, scale)
    return min(amount, cap), peril


def generate_claim_total(coverage_type, sum_assured, n_claims):
    """Aggregate severity across all claims in a policy-year."""
    if n_claims <= 0:
        return 0.0, ''
    total = 0.0
    perils = []
    for _ in range(int(n_claims)):
        amt, peril = generate_single_claim(coverage_type, sum_assured)
        total += amt
        perils.append(peril)
    return round(total, 2), '/'.join(perils)


print('Per-peril severity model ready:')
print('  Comprehensive perils:', list(PERIL_DIST['Comprehensive'].keys()))
print('  TPO perils:          ', list(PERIL_DIST['TPO'].keys()))


In [ ]:

# Retention Model (Binomial Logit Proxy)
# Probability of renewing policy next year.
# PREMIUM_CHANGE_PCT is fed from the annual portfolio trend (see cohort-simulation).
# Note: FINAL_PREMIUM_SST now evolves yearly (loadings + NCD), but the retention
# signal remains the portfolio-level trend to keep retention behavior stable.

def compute_retention_probability(row, premium_change_pct):
    """Compute probability of renewal.

    Key drivers (priority):
    1. Premium increase (highest sensitivity)
    2. Claim occurrence
    3. NCD level (incentive to stay)
    """
    p = 0.80  # Base renewal rate

    # Premium increase sensitivity (highest priority)
    if premium_change_pct > 0.15:
        p -= 0.15
    elif premium_change_pct > 0.05:
        p -= 0.10
    elif premium_change_pct < -0.05:
        p += 0.05  # Discounts improve retention

    # Claim occurrence effect
    p -= 0.25 if row.get('CLAIM_OCCURRED', False) else 0

    # NCD incentive to stay
    if row['NCD_YEARS'] >= 3:
        p += 0.15
    elif row['NCD_YEARS'] >= 2:
        p += 0.08

    return np.clip(p, 0.1, 0.95)


print('Retention model defined')
print('Base renewal rate: 80%')
print('Claim occurrence penalty: -25%')
print('Premium increase >15%: -15% (fed by annual 6% trend)')
print('NCD >=3 years bonus: +15%')


In [ ]:

# Cohort Evolution Simulation (5-year forward)
# In-force policies age each year (DRIVER_AGE, CAR_AGE +1); claim frequency and
# final premium are recomputed annually with the new ages and NCD (one-year lag:
# year N is priced with the NCD earned through year N-1).

def simulate_cohort(df_initial, n_years=5, new_entrants_per_year=5000,
                    premium_trend_annual=1.06, seed=42):
    """Simulate cohort evolution.

    Args:
        df_initial: Starting cohort (Year 1)
        n_years: Number of years to simulate
        new_entrants_per_year: New policies entering each year
        premium_trend_annual: Annual premium inflation used for retention only
        seed: Random seed for reproducibility

    Returns:
        pd.DataFrame with all policy-year records
    """
    np.random.seed(seed)
    history = []
    df_active = df_initial.copy()
    entrant_weights = (0.40, 0.40, 0.15, 0.05)
    entrant_cats = ["Gen-Z", "Millennial", "Boomers", "Senior"]

    for year_offset in range(n_years):
        year = COHORT_YEAR + year_offset
        df_active['SIM_YEAR'] = year

        # Age in-force policies (same POLID, older driver + older car).
        # Policies with COHORT_YEAR == year are brand-new entrants: keep fresh ages.
        aging_mask = df_active['COHORT_YEAR'] < year
        if aging_mask.any():
            df_active.loc[aging_mask, 'DRIVER_AGE'] += 1
            df_active.loc[aging_mask, 'CAR_AGE'] = np.minimum(
                df_active.loc[aging_mask, 'CAR_AGE'] + 1, 10
            )
            df_active.loc[aging_mask, 'DRIVER_AGE_CAT'] = df_active.loc[
                aging_mask, 'DRIVER_AGE'
            ].apply(age_to_cat)

        # Recompute frequency + premium with current ages and NCD
        # (NCD_LEVEL here still reflects claims through the PRIOR year -> one-year lag)
        df_active['CLAIM_LAMBDA'] = df_active.apply(compute_claim_lambda, axis=1)
        df_active['TOTAL_LOADING'] = df_active.apply(
            lambda r: total_loading(r['DRIVER_AGE_CAT'], r['CAR_AGE']), axis=1
        )
        df_active['NCD_LEVEL_PRICED'] = df_active['NCD_LEVEL']
        df_active['FINAL_PREMIUM_SST'] = df_active.apply(compute_final_premium, axis=1)

        # Simulate claims (Poisson frequency)
        df_active['CLAIM_COUNT'] = df_active['CLAIM_LAMBDA'].apply(
            lambda l: np.random.poisson(l)
        )
        df_active['CLAIM_OCCURRED'] = df_active['CLAIM_COUNT'] > 0

        # Simulate severity (per-peril Gamma, aggregated per policy-year)
        df_active['CLAIM_AMOUNT'] = 0.0
        df_active['CLAIM_PERIL'] = ''
        has_claims = df_active['CLAIM_OCCURRED']
        if has_claims.any():
            claimers = df_active.loc[has_claims]
            totals = claimers.apply(
                lambda r: generate_claim_total(
                    r['COVERAGE_TYPE'], r['SUM_ASSURED'], r['CLAIM_COUNT']
                ),
                axis=1
            )
            df_active.loc[has_claims, 'CLAIM_AMOUNT'] = [t[0] for t in totals]
            df_active.loc[has_claims, 'CLAIM_PERIL'] = [t[1] for t in totals]

        # Premium change signal (retention only - not stored premium)
        df_active['PREMIUM_CHANGE_PCT'] = premium_trend_annual ** year_offset - 1

        # Compute retention probability
        df_active['RENEWAL_PROB'] = df_active.apply(
            lambda r: compute_retention_probability(r, r['PREMIUM_CHANGE_PCT']),
            axis=1
        )

        # Simulate renewals
        df_active['RENEWED'] = (
            np.random.random(len(df_active)) < df_active['RENEWAL_PROB']
        )

        # Update NCD based on claims
        df_active.loc[~df_active['CLAIM_OCCURRED'], 'NCD_YEARS'] += 1
        df_active.loc[df_active['CLAIM_OCCURRED'], 'NCD_YEARS'] = 0

        # Map NCD_YEARS to NCD_LEVEL
        df_active['NCD_LEVEL'] = df_active['NCD_YEARS'].apply(
            lambda yrs: NCD_TABLE.get(min(yrs, 6), 0.55)
        )

        # Record full year state (including lapsers) for retention analysis
        cols_to_keep = ['POLID', 'COVERAGE_TYPE', 'SUM_ASSURED', 'REGION',
                        'VEHICLE_TYPE', 'DRIVER_AGE_CAT', 'DRIVER_AGE',
                        'CAR_AGE', 'DRIVER_GENDER', 'FLOOD_RISK', 'THEFT_RISK',
                        'BASIC_PREMIUM', 'FINAL_PREMIUM_SST', 'TOTAL_LOADING',
                        'NCD_LEVEL_PRICED', 'NCD_LEVEL',
                        'NCD_YEARS', 'CLAIM_LAMBDA',
                        'SIM_YEAR', 'CLAIM_COUNT', 'CLAIM_OCCURRED', 'CLAIM_AMOUNT',
                        'CLAIM_PERIL', 'PREMIUM_CHANGE_PCT',
                        'RENEWAL_PROB', 'RENEWED', 'COHORT_YEAR']
        history.append(df_active[cols_to_keep].copy())

        print(f"Year {year}: {len(df_active)} active policies, "
              f"claims: {df_active['CLAIM_COUNT'].sum()}, "
              f"freq: {df_active['CLAIM_OCCURRED'].mean():.1%}, "
              f"avg NCD priced: {df_active['NCD_LEVEL_PRICED'].mean():.2%}, "
              f"avg premium: RM{df_active['FINAL_PREMIUM_SST'].mean():.2f}, "
              f"retention: {df_active['RENEWED'].mean():.1%}")

        # Add new entrants for next year (fresh ages, unique POLID)
        if year_offset < n_years - 1 and new_entrants_per_year > 0:
            new_cohort = df_initial.sample(
                n=new_entrants_per_year, replace=True,
                random_state=seed + year_offset
            ).copy()
            n_new = len(new_cohort)
            new_cohort['POLID'] = [f"ENT{year + 1}-{i}" for i in range(n_new)]
            new_cohort['DRIVER_AGE_CAT'] = np.random.choice(
                entrant_cats, size=n_new, p=entrant_weights
            )
            new_cohort['DRIVER_AGE'] = new_cohort['DRIVER_AGE_CAT'].apply(
                generate_driver_age
            )
            new_cohort['CAR_AGE'] = new_cohort['DRIVER_AGE_CAT'].apply(
                generate_car_age
            )
            new_cohort['COHORT_YEAR'] = year + 1
            new_cohort['NCD_LEVEL'] = 0.0
            new_cohort['NCD_YEARS'] = 0
            new_cohort['BASIC_PREMIUM'] = new_cohort.apply(calculate_premium, axis=1)
            new_cohort['CLAIM_LAMBDA'] = new_cohort.apply(compute_claim_lambda, axis=1)
            new_cohort['TOTAL_LOADING'] = new_cohort.apply(
                lambda r: total_loading(r['DRIVER_AGE_CAT'], r['CAR_AGE']), axis=1
            )
            new_cohort['FINAL_PREMIUM_SST'] = new_cohort.apply(
                compute_final_premium, axis=1
            )
            df_active = pd.concat(
                [df_active[df_active['RENEWED']], new_cohort],
                ignore_index=True
            )
        else:
            df_active = df_active[df_active['RENEWED']].copy()

    return pd.concat(history, ignore_index=True)


# Run simulation (5 years)
cohort_results = simulate_cohort(df, n_years = 5 , new_entrants_per_year = int(0.25 * num_dataset))
print(f"\nSimulation complete. Total records: {len(cohort_results)}")
print(f"Year range: {cohort_results['SIM_YEAR'].min()} - {cohort_results['SIM_YEAR'].max()}")


In [ ]:
# Cohort Analysis Summary

def analyze_cohort(df_cohort):
    """Generate summary statistics from cohort simulation."""
    summary = []
    
    for year in sorted(df_cohort['SIM_YEAR'].unique()):
        year_df = df_cohort[df_cohort['SIM_YEAR'] == year]
        summary.append({
            'Year': year,
            'Active_Policies': len(year_df),
            'Total_Claims': year_df['CLAIM_COUNT'].sum(),
            'Avg_Claims_Per_Policy': year_df['CLAIM_COUNT'].mean(),
            'Total_Claim_Amount': year_df['CLAIM_AMOUNT'].sum(),
            'Avg_Claim_Amount': year_df.loc[
                year_df['CLAIM_COUNT'] > 0, 'CLAIM_AMOUNT'
            ].mean() if year_df['CLAIM_COUNT'].sum() > 0 else 0,
            'Avg_NCD_Level': year_df['NCD_LEVEL'].mean(),
            'Retention_Rate': year_df['RENEWED'].mean()
            if 'RENEWED' in year_df.columns else float('nan'),
            'Avg_Final_Premium': year_df['FINAL_PREMIUM_SST'].mean()
        })
    
    return pd.DataFrame(summary)

# Generate and display cohort summary
cohort_summary = analyze_cohort(cohort_results)
print('=== 5-Year Cohort Evolution Summary ===\n')
print(cohort_summary.to_string(index=False))
print(f"\n=== Key Findings ===")
print(f"Total claims over 5 years: {cohort_results['CLAIM_COUNT'].sum():.0f}")
print(f"Total claim cost: RM{cohort_results['CLAIM_AMOUNT'].sum():,.2f}")
print(f"Final avg NCD level: {cohort_results[cohort_results['SIM_YEAR']==2028]['NCD_LEVEL'].mean():.2%}")
print(f"Loss ratio: {cohort_results['CLAIM_AMOUNT'].sum() / cohort_results['FINAL_PREMIUM_SST'].sum():.2%}")

In [ ]:
# Test: Sample Claim Generation
# Demonstrate claim modeling on sample policies

def generate_sample_claims(df, n=5):
    """Generate sample claims for testing the model."""
    samples = df.sample(n=min(n, len(df)), random_state=42)

    print('=== Sample Claim Generation ===\n')

    for idx, row in samples.iterrows():
        lamb = row['CLAIM_LAMBDA']
        n_claims = np.random.poisson(lamb)

        print(f"Policy: {row['POLID'][:12]}...")
        print(f"  Coverage: {row['COVERAGE_TYPE']}, Vehicle: {row['VEHICLE_TYPE']}")
        print(f"  Sum Assured: RM{row['SUM_ASSURED']:,.0f}")
        print(f"  Flood Risk: {row['FLOOD_RISK']}, Theft Risk: {row['THEFT_RISK']}")
        print(f"  NCD Years: {row['NCD_YEARS']} ({row['NCD_LEVEL']:.1%})")
        print(f"  Claim Intensity (lambda): {lamb:.3f}")

        if n_claims > 0:
            total_claim, perils = generate_claim_total(
                row['COVERAGE_TYPE'], row['SUM_ASSURED'], n_claims
            )
            print(f"  Perils: {perils}")
            print(f"  TOTAL CLAIM: RM{total_claim:,.2f}")
        else:
            print('  No claims this year')

        print(f"  Basic Premium: RM{row['BASIC_PREMIUM']:,.2f}")
        print(f"  Final Premium (w/ SST): RM{row['FINAL_PREMIUM_SST']:,.2f}")
        print()


# Run sample generation
generate_sample_claims(df, n=2)


In [ ]:

# ============================================================
# Statistical Validation (Part A: Core Tests)
# ============================================================

results = cohort_results.copy()
passed = []
failed = []

def check(name, cond, detail=''):
    if cond:
        passed.append(name)
        print(f"[PASS] {name}")
    else:
        failed.append(name)
        print(f"[FAIL] {name} {detail}")

# Test 1: Overall claim frequency in plausible band (10%-20%)
overall_freq = results['CLAIM_OCCURRED'].mean()
check('1. Overall claim frequency within 10%-20%',
      0.10 <= overall_freq <= 0.20,
      f"(actual {overall_freq:.1%})")

# Test 2: TPO expected claim cost per policy-year < Comprehensive
# (frequency x mean severity - TPO has no own-damage exposure)
comp_freq_t = results.loc[results['COVERAGE_TYPE']=='Comprehensive', 'CLAIM_OCCURRED'].mean()
tpo_freq_t = results.loc[results['COVERAGE_TYPE']=='TPO', 'CLAIM_OCCURRED'].mean()
comp_sev = results.loc[(results['COVERAGE_TYPE']=='Comprehensive') &
                       (results['CLAIM_AMOUNT']>0), 'CLAIM_AMOUNT']
tpo_sev = results.loc[(results['COVERAGE_TYPE']=='TPO') &
                      (results['CLAIM_AMOUNT']>0), 'CLAIM_AMOUNT']
comp_mean = comp_sev.mean() if len(comp_sev) else 0
tpo_mean = tpo_sev.mean() if len(tpo_sev) else 0
comp_cost = comp_freq_t * comp_mean
tpo_cost = tpo_freq_t * tpo_mean
check('2. TPO expected claim cost/policy-year < Comprehensive',
      tpo_cost < comp_cost,
      f"(Comp RM{comp_cost:,.0f} vs TPO RM{tpo_cost:,.0f})")

# Test 3: Per-peril means (report; caps enforced at draw time)
peril_means = (results[results['CLAIM_AMOUNT']>0]
               .assign(peril_first=lambda d: d['CLAIM_PERIL'].str.split('/').str[0])
               .groupby('peril_first')['CLAIM_AMOUNT'].mean())
print('\n  Per-peril mean severity:')
for peril, m in peril_means.sort_values(ascending=False).items():
    print(f"    {peril:10s} RM{m:,.0f}  (n={len(results[results['CLAIM_PERIL'].str.contains(peril)])})")
print('  (caps enforced at draw time by construction)')

# Test 4: Loss ratio (incurred / earned premium) - report only
earned = results['FINAL_PREMIUM_SST'].sum()
incurred = results['CLAIM_AMOUNT'].sum()
loss_ratio = incurred / earned if earned > 0 else float('nan')
print(f"\n  Loss ratio: {loss_ratio:.1%} (earned RM{earned:,.0f}, incurred RM{incurred:,.0f})")
if not (0.40 <= loss_ratio <= 0.80):
    print(f"  [WARN] Loss ratio outside 40%-80% band - inspect premium adequacy")
else:
    print(f"  [OK]   Loss ratio within 40%-80% band")

# Test 5: NCD mechanics - claim resets to 0, claim-free increments
reset_ok = (results.loc[results['CLAIM_OCCURRED'], 'NCD_YEARS'] == 0).mean()
inc_ok = (results.loc[~results['CLAIM_OCCURRED'], 'NCD_YEARS'] >= 1).mean()
check('5a. Claims reset NCD_YEARS to 0', reset_ok >= 0.99,
      f"(reset rate {reset_ok:.1%})")
check('5b. Claim-free years increment NCD', inc_ok >= 0.99,
      f"(increment rate {inc_ok:.1%})")

# Test 6: TPO frequency < Comprehensive frequency
comp_freq = results.loc[results['COVERAGE_TYPE']=='Comprehensive', 'CLAIM_OCCURRED'].mean()
tpo_freq = results.loc[results['COVERAGE_TYPE']=='TPO', 'CLAIM_OCCURRED'].mean()
check('6. TPO claim frequency < Comprehensive',
      tpo_freq < comp_freq,
      f"(Comp {comp_freq:.1%} vs TPO {tpo_freq:.1%})")

# Test 7: Data integrity - no nulls/negatives in key columns
key_cols = ['CLAIM_COUNT', 'CLAIM_AMOUNT', 'CLAIM_LAMBDA', 'FINAL_PREMIUM_SST',
            'NCD_LEVEL', 'NCD_YEARS', 'RENEWED', 'CAR_AGE', 'TOTAL_LOADING',
            'NCD_LEVEL_PRICED']
null_bad = results[key_cols].isnull().sum().sum()
neg_bad = (results['CLAIM_AMOUNT'] < 0).sum() + (results['FINAL_PREMIUM_SST'] <= 0).sum()
check('7. No nulls in key columns', null_bad == 0, f"(nulls: {null_bad})")
check('7b. No negative/zero premium or negative claims', neg_bad == 0,
      f"(bad: {neg_bad})")

# Test 8b: Ageing works - in-force DRIVER_AGE/CAR_AGE increase across years
age_by_year = results.groupby('SIM_YEAR')[['DRIVER_AGE', 'CAR_AGE']].mean()
age_growth = age_by_year['DRIVER_AGE'].iloc[-1] > age_by_year['DRIVER_AGE'].iloc[0]
car_growth = age_by_year['CAR_AGE'].iloc[-1] > age_by_year['CAR_AGE'].iloc[0]
check('8b. Mean DRIVER_AGE rises across years', age_growth,
      f"({age_by_year['DRIVER_AGE'].iloc[0]:.1f} -> {age_by_year['DRIVER_AGE'].iloc[-1]:.1f})")
check('8c. Mean CAR_AGE rises across years', car_growth,
      f"({age_by_year['CAR_AGE'].iloc[0]:.1f} -> {age_by_year['CAR_AGE'].iloc[-1]:.1f})")

# Test 9: CAR_AGE plausibility
car_age_med = results['CAR_AGE'].median()
check('9. CAR_AGE median within 3-4 years', 3.0 <= car_age_med <= 4.0,
      f"(median {car_age_med:.1f})")
check('9b. CAR_AGE within [0,10]', results['CAR_AGE'].between(0, 10).all())
genz_car = results.loc[results['DRIVER_AGE_CAT'] == 'Gen-Z', 'CAR_AGE'].mean()
other_car = results.loc[results['DRIVER_AGE_CAT'] != 'Gen-Z', 'CAR_AGE'].mean()
check('9c. Gen-Z drive newer cars than other cohorts', genz_car < other_car,
      f"(Gen-Z {genz_car:.1f} vs others {other_car:.1f})")

# Test 10: Premium evolves across years for in-force policies
years_per_polid = results.groupby('POLID')['SIM_YEAR'].nunique()
multi_year = results[results['POLID'].isin(
    years_per_polid[years_per_polid >= 3].index
)]
prem_levels = multi_year.groupby('POLID')['FINAL_PREMIUM_SST'].nunique()
prem_varies = (prem_levels > 1).mean()
check('10. Premium varies across years for multi-year policies',
      prem_varies >= 0.9, f"({prem_varies:.1%} of policies vary)")

print(f"\n===== VALIDATION RESULT: {len(passed)} passed, {len(failed)} failed =====")


In [ ]:

# ============================================================
# FINDING 1: TPO Underpricing (documented, deliberately NOT modelled away)
# ============================================================
tpo = results.loc[results['COVERAGE_TYPE'] == 'TPO']
comp = results.loc[results['COVERAGE_TYPE'] == 'Comprehensive']
tpo_earned = tpo['FINAL_PREMIUM_SST'].sum()
tpo_incurred = tpo['CLAIM_AMOUNT'].sum()
tpo_lr = tpo_incurred / tpo_earned if tpo_earned > 0 else float('nan')
comp_earned = comp['FINAL_PREMIUM_SST'].sum()
comp_lr = comp['CLAIM_AMOUNT'].sum() / comp_earned if comp_earned > 0 else float('nan')

print('=== FINDING 1: TPO gross loss ratio ===')
print(f"TPO gross LR : {tpo_lr:.1%}  (earned RM{tpo_earned:,.0f}, incurred RM{tpo_incurred:,.0f})")
print(f"Comp gross LR: {comp_lr:.1%}")
print()
print('Root cause: TPO premium is a fixed flat amount from sources/rates.csv')
print('  (RM74-227, independent of SUM_ASSURED), while TPO claims come from the')
print('  SA-free severity model with a heavy TPBI tail (unlimited bodily injury).')
print('  => pricing inadequacy in the RATE FILE, not a claims-model defect.')
print()
print('Resolution: OUT OF SCOPE - reprice TPO Fixed Amounts in rates.csv.')
print('Deliberate choice: no artificial decline threshold, no model-side reprice,')
print('  and no tampering with the rate file.')


In [ ]:

# ============================================================
# Statistical Validation (Part B) + EDA Enrichment
# ============================================================

# Test 8: Reproducibility - re-running with same seed gives identical claims
np.random.seed(999)
run_a = simulate_cohort(df, n_years=3, seed=7)
np.random.seed(999)
run_b = simulate_cohort(df, n_years=3, seed=7)
same_claims = (run_a['CLAIM_COUNT'].values == run_b['CLAIM_COUNT'].values).all()
same_amounts = np.allclose(run_a['CLAIM_AMOUNT'].values, run_b['CLAIM_AMOUNT'].values)
if same_claims and same_amounts:
    print("[PASS] 8. Reproducibility: same seed -> identical claims")
else:
    print("[FAIL] 8. Reproducibility: seeded runs differ")

# ---- EDA enrichment ----
res = cohort_results.copy()

# E1: Loss ratio by coverage type (n/a if no earned premium)
def lr_ratio(d):
    earned = d['FINAL_PREMIUM_SST'].sum()
    if earned <= 0:
        return 'n/a (no earned premium)'
    return f"{d['CLAIM_AMOUNT'].sum() / earned:.1%}"
lr_by_cov = res.groupby('COVERAGE_TYPE').apply(lr_ratio)
print('\nE1. Loss ratio by coverage type (TPO underpriced - see FINDING 1):')
print(lr_by_cov.to_string())

# E2: Peril mix across all claims
peril_counts = res[res['CLAIM_AMOUNT'] > 0]['CLAIM_PERIL'].str.split('/').explode().value_counts()
print('\nE2. Peril mix:')
print(peril_counts.to_string())

# E3: Claim frequency by age category
freq_by_age = res.groupby('DRIVER_AGE_CAT')['CLAIM_OCCURRED'].mean().sort_values(ascending=False)
print('\nE3. Claim frequency by age category:')
print(freq_by_age.map(lambda x: f"{x:.1%}").to_string())

# E4: Retention by claim status
ret_by_claim = res.groupby('CLAIM_OCCURRED')['RENEWED'].mean()
print('\nE4. Retention by claim status:')
print(ret_by_claim.map(lambda x: f"{x:.1%}").to_string())


In [ ]:

# ============================================================
# Individual Policy Trajectories (premium evolution over years)
# ============================================================

def plot_policy_trajectories(df_cohort, n_policies=5, min_years=3,
                             out='images/policy-trajectories.png', seed=42):
    """Plot premium + priced-NCD trajectories for random multi-year policies."""
    import os
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    years_per_polid = df_cohort.groupby('POLID')['SIM_YEAR'].nunique()
    eligible = years_per_polid[years_per_polid >= min_years].index.tolist()
    rng = np.random.RandomState(seed)
    picks = [str(p) for p in rng.choice(
        eligible, size=min(n_policies, len(eligible)), replace=False
    )]

    sel = df_cohort[df_cohort['POLID'].isin(picks)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for pid in picks:
        p = sel[sel['POLID'] == pid].sort_values('SIM_YEAR')
        axes[0].plot(p['SIM_YEAR'], p['FINAL_PREMIUM_SST'], marker='o',
                     label=f"{pid[:12]}...")
        axes[1].plot(p['SIM_YEAR'], p['NCD_LEVEL_PRICED'], marker='s')
    axes[0].set_title('Final premium by year')
    axes[0].set_xlabel('SIM_YEAR')
    axes[0].set_ylabel('FINAL_PREMIUM_SST (RM)')
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)
    axes[1].set_title('NCD used for pricing by year')
    axes[1].set_xlabel('SIM_YEAR')
    axes[1].set_ylabel('NCD_LEVEL_PRICED')
    axes[1].grid(alpha=0.3)
    fig.tight_layout()

    os.makedirs(os.path.dirname(out), exist_ok=True)
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Trajectory plot saved: {out}")

    print('\n=== Selected policy trajectories (year-by-year) ===')
    cols = ['POLID', 'SIM_YEAR', 'COVERAGE_TYPE', 'DRIVER_AGE_CAT', 'DRIVER_AGE',
            'CAR_AGE', 'NCD_LEVEL_PRICED', 'FINAL_PREMIUM_SST', 'CLAIM_OCCURRED']
    print(sel.sort_values(['POLID', 'SIM_YEAR'])[cols].to_string(index=False))


plot_policy_trajectories(cohort_results)


In [ ]:
df.head()